constructed a comprehensive, multi-level feature engineering pipeline for detecting deceptive reviews.

The objective was to build independent, leakage-free signals that capture:

Linguistic patterns

User behavior

Product context

Temporal activity

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import joblib

In [2]:
df = pd.read_csv("../data/processed/trust_scored_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (719967, 29)


,user_id,product_id,rating,review_text,summary,verified,review_timestamp,reviewTime,helpful_votes,clean_review_text,...,review_day,daily_count,rule_high_frequency,rule_rating_deviation,rule_duplicate,base_trust,penalty,trust_score,fake_label,label_confidence
0,A3DDWDH9PX2YX2,7106116521,2,agree review opening small almost bent hook ex...,"I agree with the other review, the opening is ...",True,1411862400,2014-09-28,3.0,agree review opening small almost bent hook ex...,...,2014-09-28,1,0,0,0,0.593371,0.0,0.593371,0,uncertain
1,A2MWC41EW7XL15,7106116521,4,love going order another pack keep work someon...,My New 'Friends' !!,False,1408924800,2014-08-25,0.0,love going order another pack keep work someon...,...,2014-08-25,1,0,0,0,0.489583,0.0,0.489583,0,uncertain
2,A7QS961ROI6E0,7106116521,4,little plastic back work great loosing hook ea...,Works great!,True,1401494400,2014-05-31,0.0,little plastic back work great loosing hook ea...,...,2014-05-31,1,0,0,0,0.639583,0.0,0.639583,0,uncertain
3,A1BB77SEBQT8VX,B00007GDFV,3,mother law wanted present sister liked said wo...,bought as a present,True,1379808000,2013-09-22,0.0,mother law wanted present sister liked said wo...,...,2013-09-22,1,0,0,0,0.610227,0.0,0.610227,0,uncertain
4,AHWOW7D1ABO9C,B00007GDFV,3,item good quality look great fit stretched fit...,Buxton heiress collection,True,1374019200,2013-07-17,0.0,item good quality look great fit stretched fit...,...,2013-07-17,1,0,0,0,0.443561,0.0,0.443561,0,uncertain


### Basic Checks

In [3]:
print("Columns:\n", df.columns)
print("\nMissing values:\n", df.isnull().sum())

Columns:
 Index(['user_id', 'product_id', 'rating', 'review_text', 'summary', 'verified',
       'review_timestamp', 'reviewTime', 'helpful_votes', 'clean_review_text',
       'review_date', 'review_length', 'product_mean_rating',
       'rating_deviation', 'rating_score', 'user_consistency', 'helpful_ratio',
       'verified_score', 'rule_short_extreme', 'review_day', 'daily_count',
       'rule_high_frequency', 'rule_rating_deviation', 'rule_duplicate',
       'base_trust', 'penalty', 'trust_score', 'fake_label',
       'label_confidence'],
      dtype='object')

Missing values:
 user_id                    0
product_id                 0
rating                     0
review_text                0
summary                  398
verified                   0
review_timestamp           0
reviewTime                 0
helpful_votes              0
clean_review_text          0
review_date                0
review_length              0
product_mean_rating        0
rating_deviation           0
ratin

### Standardize Column Names

In [4]:
df.rename(columns={
    'clean_review_text': 'clean_review'
}, inplace=True)

### Handle Missing Values

In [5]:
df['clean_review'] = df['clean_review'].fillna("")
df['verified'] = df['verified'].fillna(0)
df['review_length'] = df['review_length'].fillna(0)
df['rating_deviation'] = df['rating_deviation'].fillna(0)

df['review_date'] = pd.to_datetime(df['review_date'])

## NLP FEATURES

### Sentiment Polarity

In [6]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

df['sentiment_score'] = df['clean_review'].apply(get_sentiment)

### Sentiment Extremeness

In [7]:
df['sentiment_extreme'] = abs(df['sentiment_score'])

### Repetition Ratio

In [8]:
def repetition_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return 1 - (len(set(words)) / len(words))

df['repetition_ratio'] = df['clean_review'].apply(repetition_ratio)

### TF-IDF (Top 5000, 1-2 grams)

In [9]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=5000,
    stop_words='english'
)

X_tfidf = tfidf.fit_transform(df['clean_review'])

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (719967, 5000)


## USER BEHAVIORAL FEATURES

### User Review Count

In [10]:
df['user_review_count'] = df.groupby('user_id')['user_id'].transform('count')

### User Rating Variance

In [11]:
df['user_rating_variance'] = df.groupby('user_id')['rating'].transform('var').fillna(0)

### User Avg Rating Deviation

In [12]:
df['user_avg_rating_deviation'] = df.groupby('user_id')['rating_deviation'].transform('mean')

### User Review Frequency

In [13]:
df['user_first_review'] = df.groupby('user_id')['review_date'].transform('min')

df['days_active'] = (df['review_date'] - df['user_first_review']).dt.days + 1

df['user_review_frequency'] = df['user_review_count'] / df['days_active']

## PRODUCT CONTEXT FEATURES

### Product Review Count

In [14]:
df['product_review_count'] = df.groupby('product_id')['product_id'].transform('count')

### Product Rating Variance

In [15]:
df['product_rating_variance'] = df.groupby('product_id')['rating'].transform('var').fillna(0)

## TEMPORAL FEATURES

### Days Since First Product Review

In [16]:
df['product_first_review'] = df.groupby('product_id')['review_date'].transform('min')

df['days_since_first_review'] = (
    df['review_date'] - df['product_first_review']
).dt.days

### Review Density

In [17]:
df['review_density'] = df['product_review_count'] / (df['days_since_first_review'] + 1)

### Burst Indicator

In [18]:
threshold = df['daily_count'].mean() + df['daily_count'].std()
df['burst_indicator'] = (df['daily_count'] > threshold).astype(int)

### Unique Word Ratio


In [19]:
def unique_word_ratio(text):
    words = str(text).split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

df['unique_word_ratio'] = df['clean_review'].apply(unique_word_ratio)


### Punctuation & Log Length


In [20]:
import numpy as np
df['exclamation_count'] = df['clean_review'].str.count('!').fillna(0)
df['question_count'] = df['clean_review'].str.count(r'\\?').fillna(0)
df['review_length_log'] = np.log1p(df['review_length'])


### Behavioral Extensions


In [21]:
df['extreme_rating'] = df['rating'].isin([1,5]).astype(int)
df['user_extreme_ratio'] = df.groupby('user_id')['extreme_rating'].transform('mean')

df['user_daily_reviews'] = df.groupby(['user_id','review_day'])['user_id'].transform('count')
df['user_burst_flag'] = (df['user_daily_reviews'] > 3).astype(int)


### Product Extensions


In [22]:
df['product_rating_std'] = df.groupby('product_id')['rating'].transform('std').fillna(0)
df['product_popularity_log'] = np.log1p(df['product_review_count'])


### Temporal Extensions


In [23]:
df = df.sort_values(['product_id','review_date'])
df['prev_review_time'] = df.groupby('product_id')['review_date'].shift(1)
df['review_time_gap'] = (df['review_date'] - df['prev_review_time']).dt.days.fillna(0)


### Graph / Diversity Extensions


In [24]:
df['user_product_diversity'] = df.groupby('user_id')['product_id'].transform('nunique')
df['product_user_diversity'] = df.groupby('product_id')['user_id'].transform('nunique')


## Select Structured Features

In [25]:
desired_features = [
    "review_length_log",
    "unique_word_ratio",
    "sentiment_score",
    "sentiment_extreme",
    "repetition_ratio",
    "hyperbolic_ratio",
    "exclamation_count",
    "question_count",
    "rating_deviation",
    "user_review_count",
    "user_rating_variance",
    "user_extreme_ratio",
    "user_review_frequency",
    "user_product_diversity",
    "product_review_count",
    "product_rating_std",
    "product_popularity_log",
    "product_user_diversity",
    "days_since_first_review",
    "review_density",
    "review_time_gap",
    "burst_indicator"
]

# Keep only features that actually exist
existing_features = [f for f in desired_features if f in df.columns]

structured_features = df[existing_features]

print("Using features:", existing_features)
print("Total structured features:", len(existing_features))

structured_features.head()

Using features: ['review_length_log', 'unique_word_ratio', 'sentiment_score', 'sentiment_extreme', 'repetition_ratio', 'exclamation_count', 'question_count', 'rating_deviation', 'user_review_count', 'user_rating_variance', 'user_extreme_ratio', 'user_review_frequency', 'user_product_diversity', 'product_review_count', 'product_rating_std', 'product_popularity_log', 'product_user_diversity', 'days_since_first_review', 'review_density', 'review_time_gap', 'burst_indicator']
Total structured features: 21


,review_length_log,unique_word_ratio,sentiment_score,sentiment_extreme,repetition_ratio,exclamation_count,question_count,rating_deviation,user_review_count,user_rating_variance,...,user_review_frequency,user_product_diversity,product_review_count,product_rating_std,product_popularity_log,product_user_diversity,days_since_first_review,review_density,review_time_gap,burst_indicator
324551,2.708050,1.000000,0.9486,0.9486,0.000000,0,84,0.000000,1,0.0,...,1.000000,1,1,0.000000,0.693147,1,0,1.00,0.0,0
381568,2.302585,1.000000,0.8625,0.8625,0.000000,0,51,0.000000,3,4.0,...,0.033333,3,1,0.000000,0.693147,1,0,1.00,0.0,1
409371,2.484907,0.909091,0.8832,0.8832,0.090909,0,86,0.000000,1,0.0,...,1.000000,1,1,0.000000,0.693147,1,0,1.00,0.0,0
315598,3.496508,0.937500,0.9562,0.9562,0.062500,0,248,0.142857,1,0.0,...,1.000000,1,7,0.377964,2.079442,7,0,7.00,0.0,0
315597,3.761200,0.904762,0.9545,0.9545,0.095238,0,278,0.142857,1,0.0,...,1.000000,1,7,0.377964,2.079442,7,3,1.75,3.0,0


### Scale Structured Features

In [26]:
scaler = StandardScaler()
X_structured = scaler.fit_transform(structured_features)

print("Structured shape:", X_structured.shape)

Structured shape: (719967, 21)


### Combine TF-IDF + Structured

In [27]:
X = hstack([X_tfidf, X_structured])
y = df['fake_label']

print("Final Feature Shape:", X.shape)

Final Feature Shape: (719967, 5021)


### Train Test Split

In [28]:
from sklearn.model_selection import train_test_split

# ==============================
# TRAIN / VALIDATION / TEST SPLIT
# ==============================

# Step 1: Train vs Temp (70 / 30)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Step 2: Validation vs Test (15 / 15)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# ==============================
# SHAPE CHECK
# ==============================

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

# ==============================
# DATA DISTRIBUTION CHECK
# ==============================

print("\nTrain label distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nValidation label distribution:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nTest label distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train shape: (503976, 5021)
Validation shape: (107995, 5021)
Test shape: (107996, 5021)

Train label distribution:
fake_label
0    467008
1     36968
Name: count, dtype: int64
fake_label
0    0.926647
1    0.073353
Name: proportion, dtype: float64

Validation label distribution:
fake_label
0    100073
1      7922
Name: count, dtype: int64
fake_label
0    0.926645
1    0.073355
Name: proportion, dtype: float64

Test label distribution:
fake_label
0    100074
1      7922
Name: count, dtype: int64
fake_label
0    0.926645
1    0.073355
Name: proportion, dtype: float64


### Feature Correlation Analysis & Pruning
Identify and remove highly correlated features (>0.85) to reduce multicollinearity and redundancy.

In [29]:
print("Computing correlation matrix matrix...")
# Select only numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

# Find highly correlated features
high_corr_threshold = 0.85
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column].abs() > high_corr_threshold)]

print("\nHighly Correlated Feature Pairs (Absolute Correlation > 0.85):")
for col in to_drop:
    correlated_with = upper_triangle.index[upper_triangle[col].abs() > high_corr_threshold].tolist()
    print(f"- '{col}' is highly correlated with: {correlated_with}")

print(f"\nDropping {len(to_drop)} highly correlated features: {to_drop}")
df = df.drop(columns=to_drop)
print(f"New dataset shape: {df.shape}")

Computing correlation matrix matrix...

Highly Correlated Feature Pairs (Absolute Correlation > 0.85):
- 'rating_score' is highly correlated with: ['rating_deviation']
- 'trust_score' is highly correlated with: ['base_trust']
- 'user_rating_variance' is highly correlated with: ['user_consistency']
- 'user_avg_rating_deviation' is highly correlated with: ['rating_deviation', 'rating_score']
- 'burst_indicator' is highly correlated with: ['daily_count']
- 'unique_word_ratio' is highly correlated with: ['repetition_ratio']
- 'question_count' is highly correlated with: ['review_length']
- 'user_extreme_ratio' is highly correlated with: ['extreme_rating']
- 'user_daily_reviews' is highly correlated with: ['daily_count', 'burst_indicator']
- 'user_burst_flag' is highly correlated with: ['rule_high_frequency']
- 'product_rating_std' is highly correlated with: ['product_rating_variance']
- 'user_product_diversity' is highly correlated with: ['user_review_count']
- 'product_user_diversity' is h

### Save Artifacts

In [30]:
os.makedirs("../models", exist_ok=True)
df.to_csv("../data/processed/featured_dataset.csv", index=False)

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
joblib.dump(scaler, "../models/feature_scaler.pkl")

print("Artifacts saved.")

Artifacts saved.
